In [1]:
import sys
import os

sys.path.append(os.path.abspath(".."))

In [2]:
from scripts.training.preprocessed import get_final_data
from scripts.training.regression import train_regression

In [3]:
train_df, test_df = get_final_data(data_dir="../data", verbose=False, encoder_dir="../encoders")

train_cols = set(train_df.columns)
test_cols = set(test_df.columns)

print(f"Onlt in test: {test_cols - train_cols}")
print(f"Only in train: {train_cols - test_cols}")
print(f"Common cols: {len(train_cols & test_cols)}")

Обнаружено 4023 записей с дублирующимися названиями после чистки!
Примеры дубликатов:
    item_id                      item_name
12       12  МИХЕЙ И ДЖУМАНДЖИ Сука любовь
30       30        007 КООРДИНАТЫ СКАЙФОЛЛ
31       31        007 КООРДИНАТЫ СКАЙФОЛЛ
32       32                             11
33       33                             11
35       35                  10 ЛЕТ СПУСТЯ
36       36                  10 ЛЕТ СПУСТЯ
37       37                  10 ЛЕТ СПУСТЯ
71       71            11 ДРУЗЕЙ ОУШЕНА WB
72       72            11 ДРУЗЕЙ ОУШЕНА WB

Найдено 1657 уникальных названий, которые повторяются:
                       item_name  count
0        007 КООРДИНАТЫ СКАЙФОЛЛ      2
1                  10 ЛЕТ СПУСТЯ      3
2                             11      2
3            11 ДРУЗЕЙ ОУШЕНА WB      2
4            12 ДРУЗЕЙ ОУШЕНА WB      2
5                 12 ЛЕТ РАБСТВА      2
6                      127 ЧАСОВ      3
7                   12ДВЕНАДЦАТЬ      2
8            13 ДРУЗЕЙ ОУ

In [9]:
for split in [25, 28, 30, 32]:
    print(f"\n--- split_month = {split} ---")
    train_regression(train_df, test_df, split_month=split)
    print(f"-"*50 + "\n\n")


--- split_month = 25 ---
Training until validation scores don't improve for 20 rounds
[50]	valid_0's rmse: 2.07474	valid_0's tweedie: 5.36618
[100]	valid_0's rmse: 1.89408	valid_0's tweedie: 5.31113
[150]	valid_0's rmse: 1.76852	valid_0's tweedie: 5.28875
[200]	valid_0's rmse: 1.6898	valid_0's tweedie: 5.2787
[250]	valid_0's rmse: 1.64343	valid_0's tweedie: 5.27345
[300]	valid_0's rmse: 1.62042	valid_0's tweedie: 5.27135
[350]	valid_0's rmse: 1.61024	valid_0's tweedie: 5.27035
[400]	valid_0's rmse: 1.60225	valid_0's tweedie: 5.26946
[450]	valid_0's rmse: 1.5985	valid_0's tweedie: 5.26905
[500]	valid_0's rmse: 1.59613	valid_0's tweedie: 5.26852
Early stopping, best iteration is:
[527]	valid_0's rmse: 1.59511	valid_0's tweedie: 5.26829
RMSE регрессора (только на положительных clipped): 1.5951
--------------------------------------------------



--- split_month = 28 ---
Training until validation scores don't improve for 20 rounds
[50]	valid_0's rmse: 2.0832	valid_0's tweedie: 5.35319
[1

In [4]:
import pandas as pd
import numpy as np
import lightgbm as lgb

def make_submission(train_df: pd.DataFrame, test_df: pd.DataFrame, split_month: int = 33):
    
    train_mask = train_df['date_block_num'] < split_month
    
    common_cols = [col for col in train_df.columns if col in test_df.columns and col != 'item_cnt_month']

    X_train = train_df[train_mask][common_cols]
    y_train = np.clip(train_df[train_mask]['item_cnt_month'], 0, 20)
    X_test = test_df[common_cols]
    
    cat_cols = ["global_category", "shop_city"]
    for col in cat_cols:
        X_train[col] = X_train[col].astype('category')
        X_test[col] = X_test[col].astype('category')
    
    model = lgb.LGBMRegressor(
        objective='tweedie',
        tweedie_variance_power=1.1,
        n_estimators=1000,
        learning_rate=0.01,
        max_depth=12,
        num_leaves=64,
        random_state=42,
        verbosity=-1
    )
    
    model.fit(X_train, y_train, categorical_feature=cat_cols)
    
    pred = np.clip(model.predict(X_test), 0, 20)
    
    sample_sub = pd.read_csv('../data/sample_submission.csv')
    sample_sub['item_cnt_month'] = pred
    sample_sub.to_csv('submission_kaggle.csv', index=False)
    print("✅ Сабмит сохранён в submission_kaggle.csv")
    return pred

In [5]:
make_submission(train_df, test_df, split_month=25)

✅ Сабмит сохранён в submission_kaggle.csv


array([2.73322939, 0.91649176, 1.30845071, ..., 1.04226101, 1.14705465,
       1.35616129], shape=(214200,))